In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

# Dataset paths
train_dir = "/content/Training"
test_dir = "/content/Testing"

# Image settings
IMG_SIZE = 224
BATCH_SIZE = 32

# Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Load training data
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Load testing data
test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Load pretrained MobileNetV2
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze pretrained layers
base_model.trainable = False

# Build final model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(4, activation='softmax')
])

# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train model
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=5
)

# Save model
model.save("brain_model.h5")

print("Model saved successfully!")

# Evaluate model
loss, accuracy = model.evaluate(test_data)

print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Test Loss: {loss:.4f}")

# Plot graph
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
from google.colab import files
files.download("brain_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
import os

# Load model
model = tf.keras.models.load_model("brain_model.h5")

# Class labels
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

# List uploaded images
image_files = [
    "Te-aug-me_7.jpg",
    "Te-gl_6.jpg",
    "Te-pi_14.jpg",
    "Te-pi_6.jpg"
]

# Predict each image
for img_name in image_files:

    img = image.load_img(img_name, target_size=(224, 224))

    img_array = image.img_to_array(img)

    img_array = img_array / 255.0

    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)

    predicted_index = np.argmax(prediction)

    predicted_class = class_names[predicted_index]

    confidence = np.max(prediction) * 100

    print(f"{img_name} --> {predicted_class} ({confidence:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step
Te-aug-me_7.jpg --> meningioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Te-gl_6.jpg --> glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Te-pi_14.jpg --> pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Te-pi_6.jpg --> pituitary
